# Why Is Measles Back?

**A data story on US measles resurgence and the geography of vaccination coverage**

*PSYC 81.09: Storytelling with Data — Spring 2026*
*Dartmouth College | Professor Jeremy Manning*

**Author:** Sam Macuga

**Video:** https://youtu.be/myMGMW1ZCe4

---

# Background and overview

**Research question.** The United States declared measles eliminated in 2000, after near-total vaccination coverage drove confirmed cases under 100 per year. In 2025, the US recorded 2,288 cases — the most in any year since 1992 — and the country is now under formal review for whether it still qualifies as having eliminated the disease. The vaccine hasn't changed. So what has?

This notebook argues that the answer isn't visible in a single national coverage number. It's a structural change in *where* the coverage gaps live — and what kind of disease environment that geography creates.

## Project team

Sole author: Sam Macuga (Dartmouth College, Storytelling with Data, PSYC 81.09).

## Quick summary of findings

1. **National coverage tells an incomplete story.** US kindergarten MMR coverage dropped from 95.2% (2019–20) to 92.5% (2024–25) — about 3 percentage points. But that small national change conceals a structural shift: the *number of states* with substantially low coverage has risen sharply.

2. **The geography of vulnerability has reorganized.** From 2011 through 2019, only 3–4 states had kindergarten MMR coverage below 90%. By the 2024–25 school year, 16 do. The pre-pandemic distribution was a stable handful of low-coverage outliers; the current distribution is widespread sub-threshold coverage.

3. **Cases followed coverage, but with a delay.** Coverage began falling in 2020–21, but case counts stayed relatively low through 2024. In early 2025, an outbreak in a low-coverage Mennonite community in West Texas sparked sustained transmission that spread to New Mexico and other states. The conditions for an outbreak had been building for years; the spark and the cluster finally connected.

## Approach

I combined two CDC data streams for the period 2010–2026:

- **Confirmed measles cases per year** from the CDC's Measles Cases and Outbreaks reporting page.
- **Kindergarten MMR vaccination coverage by state** from the CDC's SchoolVaxView program, supplemented by annual MMWR reports.

For each year of vaccination data, I categorized the 50 states + DC into three tiers based on their MMR coverage: ≥95% (at the herd immunity threshold), 90–94.99% (below threshold but moderate), and <90% (substantially below). The tier-based approach makes the geographic *distribution* of coverage legible in a way that a single national average can't.

---

# Data

## Sources

| Source | Data | Coverage period |
| --- | --- | --- |
| CDC Measles Cases and Outbreaks | Confirmed annual case counts | 2010–2026 (2026 YTD: May 21) |
| CDC SchoolVaxView | Kindergarten MMR coverage by state | 2009–10 through 2024–25 school years |
| CDC MMWR annual reports | State-level breakdowns of coverage tiers | 2011–12 through 2023–24 school years |
| KFF analyses | Cross-year comparisons | 2019–20 vs 2024–25 school years |

## Notes on data availability

The CDC's standardized kindergarten coverage data begins with the 2009–10 school year. Pre-2010 coverage estimates exist via the National Immunization Survey, but they sample a different age cohort (19–35-month-olds) and aren't directly comparable to the school-entry data used here. This analysis starts in 2010 to keep the two metrics — cases and coverage — on the same time scale.

For some intermediate years (2013–2017), the CDC published median coverage and national totals but did not publish the precise count of states in each tier. Where exact tier counts weren't available, I interpolated between verified endpoints and flagged those rows in the CSV with `data_quality = interpolated`. The overall trend — stable through 2019, sharp post-pandemic deterioration — is supported by the verified endpoints; year-to-year intermediate values should be treated as approximate.

## Data files

This notebook reads from two CSV files in `data/`:

- `data/measles_cases_by_year.csv` — annual confirmed case counts, 2010–2026.
- `data/state_mmr_coverage_tiers.csv` — count of jurisdictions in each coverage tier, by school year ending in the given calendar year.

---

## Data loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from pathlib import Path

# --- Locate the data files --------------------------------------------------
# The notebook normally lives next to a `data/` folder containing the two CSVs.
# In Colab, files may land in /content instead, so we search a few sensible
# locations rather than assuming the working directory is the project root.
def _find_data_dir():
    candidates = [
        Path("data"),                            # notebook's own folder
        Path.cwd() / "data",                     # current working dir
        Path("/content/data"),                   # Colab default
        Path("/content/measles_story/data"),     # Colab, if folder uploaded
        Path.home() / "measles_story" / "data",  # local clone
    ]
    for c in candidates:
        if (c / "measles_cases_by_year.csv").exists():
            return c
    raise FileNotFoundError(
        "Could not find measles_cases_by_year.csv. Searched:\n  "
        + "\n  ".join(str(c) for c in candidates)
        + "\n\nIn Colab: upload measles_cases_by_year.csv and "
        "state_mmr_coverage_tiers.csv into a `data/` folder, "
        "or set DATA = Path('/content') after uploading them to the root."
    )

DATA = _find_data_dir()
print(f"Loading from: {DATA}")

cases_df = pd.read_csv(DATA / "measles_cases_by_year.csv")
coverage_df = pd.read_csv(DATA / "state_mmr_coverage_tiers.csv")

print(f"Cases:    {len(cases_df)} years ({cases_df['year'].min()}-{cases_df['year'].max()})")
print(f"Coverage: {len(coverage_df)} school years "
      f"({coverage_df['school_year_end'].min()}-{coverage_df['school_year_end'].max()})")

# --- Color palette (used by all figures in this notebook) -------------------
# Matches the simulation HTML used in the accompanying video.
BG     = "#f8f4e8"   # paper cream
TEXT   = "#2a2520"   # near-black warm
MUTED  = "#7a6f60"   # muted brown
LINE   = "#d4cab5"   # pale stone
RECOV  = "#7a3b69"   # plum — outbreak years
IMM    = "#0a7e8c"   # teal — at/above threshold
WARN   = "#c9930b"   # mustard — between thresholds
DANGER = "#c1272d"   # red — substantially below


In [ ]:
# Quick peek at the case series
cases_df.head(3)

In [ ]:
# And the coverage tiers
coverage_df.head(3)

### Sanity checks

Each row of coverage data should sum to either 50 (states only) or 51 (states + DC), depending on whether DC was reported separately in that year's MMWR. Verify before plotting.

In [ ]:
tier_sum = (coverage_df['states_at_or_above_95']
            + coverage_df['states_90_to_95']
            + coverage_df['states_below_90'])
mismatch = coverage_df.loc[tier_sum != coverage_df['total_jurisdictions']]
assert mismatch.empty, f"Tier sums don't match totals: {mismatch}"

print("All tier sums match their reported total_jurisdictions column.")
row_2019 = coverage_df.loc[coverage_df.school_year_end == 2019].iloc[0]
row_2025 = coverage_df.loc[coverage_df.school_year_end == 2025].iloc[0]
print()
print(f"2018-19 school year: {row_2019.states_at_or_above_95} states >=95%, "
      f"{row_2019.states_below_90} states <90%")
print(f"2024-25 school year: {row_2025.states_at_or_above_95} states >=95%, "
      f"{row_2025.states_below_90} states <90%")

---

# Visual: how measles compares to other infectious diseases

Before getting to the simulation and the national case data, it helps to ground one fact: how transmissible measles actually is relative to other diseases people have heard of. The chart below uses the basic reproduction number ($R_0$) — the average number of new infections one infected person produces in a fully susceptible population. Measles' $R_0$ is between 12 and 18, several times higher than COVID-19 (around 2.5–3.5) and an order of magnitude higher than seasonal flu (around 1.2–1.4).

Sources: Guerra et al. (2017) *Human Vaccines & Immunotherapeutics* for measles; CDC and WHO published systematic reviews for the others.


In [ ]:
# ── R0 comparison chart ──────────────────────────────────────────────────────
# Uses the same palette as the composite figure (defined earlier).

# R0 ranges per disease, ordered top-to-bottom by highest R0 first.
diseases = [
    ("Measles",      12.0, 18.0, "12–18",   True),
    ("Chickenpox",   10.0, 12.0, "10–12",   False),
    ("Smallpox",      5.0,  7.0, "5–7",     False),
    ("COVID-19",      2.5,  3.5, "2.5–3.5", False),
    ("Ebola",         1.5,  2.5, "1.5–2.5", False),
    ("Seasonal flu",  1.2,  1.4, "1.2–1.4", False),
]

# Colors local to this figure
BAR  = "#a39786"   # warm taupe for non-measles bars
RED  = DANGER     # reuse the project's red

n = len(diseases)
y_positions = list(range(n - 1, -1, -1))   # top-down (measles at top)

fig, ax = plt.subplots(figsize=(13, 7.5), dpi=150)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

for i, (name, lo, hi, label, is_measles) in enumerate(diseases):
    y = y_positions[i]
    color = RED if is_measles else BAR
    ax.barh(y, hi - lo, left=lo, height=0.55, color=color,
            edgecolor="none", zorder=3)
    # Range label to the right of the bar
    ax.text(hi + 0.3, y, label, va="center", ha="left",
            fontsize=14 if is_measles else 13,
            color=TEXT if is_measles else MUTED,
            fontweight="bold" if is_measles else "normal")
    # Disease name on the left
    ax.text(-0.7, y, name, va="center", ha="right",
            fontsize=16 if is_measles else 15,
            color=TEXT,
            fontweight="bold" if is_measles else "normal")

ax.set_xlim(-0.5, 22)
ax.set_ylim(-0.7, n - 0.3)
ax.set_yticks([])
ax.set_xticks([0, 5, 10, 15, 20])
ax.tick_params(axis="x", colors=MUTED, labelsize=12, length=0, pad=8)

for spine in ("top", "right", "left"):
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color(LINE)
ax.spines["bottom"].set_linewidth(1)

ax.text(11, -1.45,
        r"Basic reproduction number ($R_0$):  average new infections per case in a fully susceptible population",
        ha="center", va="top", fontsize=12, color=MUTED)

fig.text(0.04, 0.93,
         r"Estimated $\mathbf{R_0}$ for select infectious diseases. Higher = more contagious.",
         color=TEXT, fontsize=18, fontweight="bold", ha="left", va="top")

fig.text(0.04, 0.04,
         "Sources: Guerra et al. 2017 (measles); CDC; WHO; published systematic reviews.",
         color=MUTED, fontsize=10, fontstyle="italic", ha="left", va="bottom")

plt.subplots_adjust(left=0.16, right=0.96, top=0.85, bottom=0.18)
plt.savefig("r0_comparison.png", dpi=150, facecolor=BG, bbox_inches=None)
plt.show()

---

# Analysis

The analysis has two parts. First, we run the same neighbor-based outbreak model that drives the companion HTML visualization — but instead of playing back a single canned run, we run it a thousand times for each state and look at the distribution of outcomes. This lets us check whether the "Connecticut contains it / Idaho spreads it" pattern holds in general, or whether the single canned run happens to be unrepresentative.

Second, we plot the national picture: confirmed measles cases per year alongside how the fifty states + DC are distributed across MMR coverage tiers.

## Outbreak simulation: Connecticut vs Idaho

The model is a stochastic neighbor-based SEIR on a 10×10 grid (100 people), parameters from the companion `measles_sim.html`: latent period 10 days, infectious period 8 days, MMR efficacy 97%, and 3 unvaccinable individuals (newborns, immunocompromised, etc.) placed randomly each run. The only difference between the two scenarios is the kindergarten MMR coverage rate — 98% for Connecticut (the highest in the country), 78.5% for Idaho (the lowest). We seed a single random susceptible person as patient zero and let the outbreak play out, then record how many people got sick.

In [ ]:
# Import the simulation module (see seir_simulation.py for the model definition)
# Find seir_simulation.py in a few sensible locations so this works whether you
# run the notebook from the project folder or in Colab.
import sys
from pathlib import Path

_sim_candidates = [
    Path("."),
    Path.cwd(),
    Path("/content"),
    Path("/content/measles_story"),
    Path.home() / "measles_story",
]
for _p in _sim_candidates:
    if (_p / "seir_simulation.py").exists():
        sys.path.insert(0, str(_p))
        break
else:
    raise FileNotFoundError(
        "Could not find seir_simulation.py. Place it in the same folder as this "
        "notebook (or upload it to /content in Colab)."
    )

from seir_simulation import run_many

N_RUNS = 1000  # outbreaks per scenario

ct_totals, ct_vulns = run_many(coverage=0.980, n_runs=N_RUNS, seed=1)
id_totals, id_vulns = run_many(coverage=0.785, n_runs=N_RUNS, seed=2)

print(f"Ran {N_RUNS:,} outbreaks per scenario.\n")
print(f"Connecticut (98.0% coverage):")
print(f"  Median outbreak size: {int(np.median(ct_totals)):>3} people")
print(f"  Mean outbreak size:   {ct_totals.mean():>5.1f} people")
print(f"  90th percentile:      {int(np.percentile(ct_totals, 90)):>3} people")
print(f"  Largest outbreak:     {ct_totals.max():>3} people")
print(f"  P(outbreak ≥ 5):      {np.mean(ct_totals >= 5):>6.1%}")
print()
print(f"Idaho (78.5% coverage):")
print(f"  Median outbreak size: {int(np.median(id_totals)):>3} people")
print(f"  Mean outbreak size:   {id_totals.mean():>5.1f} people")
print(f"  90th percentile:      {int(np.percentile(id_totals, 90)):>3} people")
print(f"  Largest outbreak:     {id_totals.max():>3} people")
print(f"  P(outbreak ≥ 5):      {np.mean(id_totals >= 5):>6.1%}")


In [ ]:
# Plot the outbreak size distributions side by side

fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

bins = np.arange(0, max(ct_totals.max(), id_totals.max()) + 2) - 0.5

ax.hist(ct_totals, bins=bins, color=IMM, alpha=0.7, edgecolor='none',
        label=f'Connecticut, 98.0% coverage  (median {int(np.median(ct_totals))})', zorder=3)
ax.hist(id_totals, bins=bins, color=DANGER, alpha=0.65, edgecolor='none',
        label=f'Idaho, 78.5% coverage  (median {int(np.median(id_totals))})', zorder=3)

ax.axvline(np.median(ct_totals), color=IMM, linestyle='--', linewidth=1.2, alpha=0.8, zorder=4)
ax.axvline(np.median(id_totals), color=DANGER, linestyle='--', linewidth=1.2, alpha=0.8, zorder=4)

ax.set_xlabel('Total people infected per outbreak', color=TEXT, fontsize=11)
ax.set_ylabel(f'Number of outbreaks (out of {N_RUNS:,})', color=TEXT, fontsize=11)
ax.set_title(f'Outbreak size distribution over {N_RUNS:,} runs of the same model',
             color=TEXT, fontsize=13, fontweight='bold', loc='left', pad=12)
ax.tick_params(colors=TEXT, labelsize=10, length=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color(LINE)
ax.spines['bottom'].set_color(LINE)
ax.legend(loc='upper right', frameon=False, fontsize=10, labelcolor=TEXT)
ax.set_xlim(-0.5, max(ct_totals.max(), id_totals.max()) + 0.5)

plt.tight_layout()
plt.savefig('outbreak_distributions.png', dpi=150, facecolor=BG, bbox_inches='tight')
plt.show()

**What the distributions show.** In Connecticut, the typical outbreak ends after one person — patient zero infects nobody else and the chain dies immediately. Even at the 90th percentile, only three people are infected. About 2% of runs produce an outbreak of five or more.

In Idaho, the typical outbreak reaches five people, the 90th percentile reaches thirteen, and the worst run in this batch infected twenty-six. About half of Idaho runs produce an outbreak of five or more.

The two distributions overlap at the low end — sometimes Idaho's chain happens to die quickly, just by luck of where patient zero lands. But Idaho's distribution has a long right tail that Connecticut's doesn't. The single canned run shown in the HTML video (Connecticut: 1 infected, Idaho: 18 infected) is at the modal value for Connecticut but near the 95th percentile for Idaho — dramatic but not typical. The Monte Carlo distribution is the honest version of the comparison: the gap between scenarios is real and substantial, but the within-scenario variability matters too.

This is why epidemiologists talk about the *probability* of sustained transmission rather than a deterministic outcome. With each percentage point of coverage you lose, the right tail of the outbreak-size distribution gets fatter.

## The national picture

The figure below pairs the two CDC data streams. The top panel shows confirmed measles cases each year since 2010. The bottom panel shows where the 50 states + DC sit relative to the 95% herd immunity threshold for kindergarten MMR coverage.

In [ ]:
# ── Composite figure: cases above, state coverage tiers below ────────────────

years_cases = cases_df["year"].to_numpy()
cases       = cases_df["confirmed_cases"].to_numpy()

years_cov  = coverage_df["school_year_end"].to_numpy()
above_95   = coverage_df["states_at_or_above_95"].to_numpy()
between    = coverage_df["states_90_to_95"].to_numpy()
below_90   = coverage_df["states_below_90"].to_numpy()

fig = plt.figure(figsize=(13, 8.5), dpi=150)
fig.patch.set_facecolor(BG)
gs = gridspec.GridSpec(2, 1, height_ratios=[2.2, 1.0], hspace=0.32,
                       left=0.08, right=0.96, top=0.89, bottom=0.09)

# ── TOP: cases per year ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor(BG)

bar_colors = []
for y in years_cases:
    if y == 2019 or y >= 2025:
        bar_colors.append(RECOV)        # near-miss + record years
    elif y in (2020, 2021):
        bar_colors.append(LINE)         # pandemic dip — visually muted
    else:
        bar_colors.append(MUTED)        # elimination-era baseline

ax1.bar(years_cases, cases, color=bar_colors, width=0.72, edgecolor="none", zorder=3)
ax1.set_xlim(2009.2, 2026.8)
ax1.set_ylim(0, 2700)
ax1.set_xticks(range(2010, 2027, 2))
ax1.set_yticks(range(0, 2501, 500))
ax1.set_yticklabels([f"{y:,}" for y in range(0, 2501, 500)])
ax1.tick_params(colors=TEXT, labelsize=10, length=0)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.spines["left"].set_color(LINE)
ax1.spines["bottom"].set_color(LINE)
ax1.set_ylabel("Confirmed measles cases", color=TEXT, fontsize=11, labelpad=10)

ax1.annotate("2019: 1,274 cases\nnearly lost elimination status",
             xy=(2019, 1274), xytext=(2014.5, 700),
             color=RECOV, fontsize=10, ha="center",
             arrowprops=dict(arrowstyle="->", color=RECOV, lw=1, alpha=0.7))
ax1.text(2020.5, 200, "Pandemic dip", color=MUTED, fontsize=9,
         ha="center", style="italic", alpha=0.9)
ax1.text(2025, 2288 + 70, "2,288", color=RECOV, fontsize=11,
         fontweight="bold", ha="center")
ax1.text(2026, 1952 + 70, "1,952", color=RECOV, fontsize=11,
         fontweight="bold", ha="center")

# ── BOTTOM: state coverage tiers (stacked bars) ──────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor(BG)

bar_w = 0.72

ax2.bar(years_cov, below_90, width=bar_w, color=DANGER, edgecolor="none", zorder=3)
ax2.bar(years_cov, between, width=bar_w, bottom=below_90, color=WARN,
        edgecolor="none", zorder=3, alpha=0.85)
ax2.bar(years_cov, above_95, width=bar_w, bottom=below_90 + between,
        color=IMM, edgecolor="none", zorder=3, alpha=0.85)

ax2.set_xlim(2009.2, 2026.8)
ax2.set_ylim(0, 55)
ax2.set_xticks(range(2010, 2027, 2))
ax2.set_yticks([0, 10, 20, 30, 40, 50])
ax2.tick_params(colors=TEXT, labelsize=10, length=0)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.spines["left"].set_color(LINE)
ax2.spines["bottom"].set_color(LINE)
ax2.set_ylabel("Number of jurisdictions\n(50 states + DC)",
               color=TEXT, fontsize=10, labelpad=10)

legend_elements = [
    mpatches.Patch(color=IMM, alpha=0.85,
                   label="≥95% (at or above herd immunity threshold)"),
    mpatches.Patch(color=WARN, alpha=0.85,
                   label="90–94.99% (below threshold)"),
    mpatches.Patch(color=DANGER,
                   label="<90% (substantially below)"),
]
ax2.legend(handles=legend_elements, loc="upper left",
           bbox_to_anchor=(0, 1.18), ncol=3, frameon=False,
           fontsize=9.5, labelcolor=TEXT, handlelength=1.3,
           handleheight=1.0, columnspacing=1.4)

fig.suptitle("US measles cases and the geography of vaccination coverage, 2010–2026",
             color=TEXT, fontsize=15, fontweight="bold",
             x=0.08, y=0.96, ha="left")
fig.text(0.08, 0.925,
         "Top: confirmed cases per year. Bottom: # of states with kindergarten MMR coverage by tier.",
         color=MUTED, fontsize=10.5, ha="left")
fig.text(0.08, 0.015,
         "Sources: CDC Measles Cases and Outbreaks; CDC SchoolVaxView MMWR reports (2011–12 through 2024–25); "
         "KFF analyses. 2026 cases through May 21, 2026.",
         color=MUTED, fontsize=8.5, ha="left", style="italic")

plt.savefig("composite.png", dpi=150, facecolor=BG, bbox_inches="tight")
plt.show()

## Interpreting the figure

### The elimination era held through 2019

For most of the 2010s, the bottom panel is remarkably stable. About 22 states stayed at ≥95% MMR coverage every year, about 26 sat in the 90–94.99% range, and only 3–4 were below 90%. The composition shifted slightly year to year, but the *distribution* held steady. Cases in the top panel reflect this: most years under 200, with travelers occasionally importing chains that fizzled out.

The 2014 spike (667 cases) was driven primarily by a Disneyland-linked outbreak that reached unvaccinated communities. The 2019 spike (1,274 cases) was driven by sustained transmission in undervaccinated Orthodox Jewish communities in Brooklyn and Rockland County, NY. In both cases, **national coverage was still at or near 95%** — which is exactly the point. National coverage doesn't predict outbreaks. Outbreaks happen where unvaccinated people cluster locally, and those clusters can exist inside a country whose average looks fine.

### The pandemic broke the distribution

In the 2020–21 school year, all four CDC-tracked childhood vaccinations (MMR, DTaP, polio, varicella) declined about one percentage point nationally. The next year they declined another point, and the trajectory hasn't reversed since. By the 2024–25 school year, kindergarten MMR coverage was 92.5%, down from 95.2% pre-pandemic.

The bottom panel makes the geographic story visible. The teal segment (states ≥95%) shrank from 23 in 2019 to 11 in 2024 and 2025. The red segment (states <90%) grew from 3 to 16 over the same period. This isn't a uniform 3-point drop everywhere — it's a redistribution. States that were stable at 94–96% mostly held; states that had been borderline drifted below 90%. The country went from "a stable handful of low-coverage outliers" to "widespread sub-threshold coverage."

### Cases lagged the coverage shift

Look at 2022, 2023, and 2024: case counts stayed under 300 even as the bottom panel deteriorated. Measles outbreaks need a spark — an imported case that lands in an undervaccinated community and sustains transmission. The conditions were building, but the spark and the cluster hadn't connected at scale.

That changed in early 2025. A measles case in a tight-knit Mennonite community in West Texas — a community with very low vaccination coverage — started a transmission chain that lasted months. It spread to New Mexico and seeded further outbreaks in other states. By year-end, the US had recorded 2,288 cases, three deaths (two children in Texas, one adult in New Mexico), and 49 distinct outbreaks. The 2026 year-to-date count was already 1,952 cases by late May.

### What this means for elimination status

The WHO defines measles "elimination" as the absence of continuous endemic transmission for at least 12 consecutive months. The 2025 US outbreaks involved several chains that lasted close to but not quite 12 months. If transmission of the same strain continues into 2026 for more than 12 months uninterrupted, the US loses elimination status.

In November 2025, Canada lost its elimination status for exactly this reason, and the WHO Region of the Americas lost its regional measles-free designation as a result. The US individual status review is scheduled for November 2026.

---

# Future directions

A few extensions would deepen this analysis:

- **County-level data.** State-level averages still hide local clusters. A county- or zip-code-level extension of the bottom panel would show where the genuinely high-risk communities sit. The CDC publishes county-level data for some states but not others; assembling a national series would require pulling from individual state health departments. NBC News and Stanford researchers built this for Texas in 2025 (139 of 254 counties below 95%); a national version doesn't yet exist.

- **Linking outbreaks to coverage geography.** A spatially-explicit analysis comparing outbreak locations to the coverage map would test whether the 2025 outbreaks did originate in the lowest-coverage clusters. Anecdotal evidence (the Texas Mennonite community, Idaho exemption-driven communities) supports this, but a systematic mapping would be more rigorous.

- **Exemption type breakdowns.** The post-pandemic decline isn't just "missed appointments" — it includes a documented rise in non-medical exemptions, particularly philosophical and religious ones. A breakdown of below-threshold states by exemption regime would help separate "still catching up from pandemic" from "permanent policy shift."

- **Hesitancy versus access.** Coverage can drop for reasons that look identical in aggregate data: structural access problems (no pediatrician available, lost insurance), or vaccine refusal. These have very different policy implications. A survey-based extension could distinguish them.

---

# Acknowledgments

Thanks to Professor Jeremy Manning and PSYC 81.09 (Storytelling with Data) at Dartmouth College.

This analysis was assisted by Anthropic's Claude (Claude Opus 4.7) for data verification, figure design, and notebook structure. Anthropic's Claude was also used to support the writing of the accompanying video narration script. All analytical decisions, final narrative choices, source verification, and interpretations are my own.

**Data sources:**
- CDC, *Measles Cases and Outbreaks*: https://www.cdc.gov/measles/data-research/index.html
- CDC, *SchoolVaxView*: https://www.cdc.gov/schoolvaxview/data/index.html
- CDC MMWR, *Vaccination Coverage Among Kindergartners* (annual reports, 2011–12 through 2023–24 school years)
- KFF, *Kindergarten Routine Vaccination Rates Continue to Decline* (September 2025): https://www.kff.org/medicaid/kindergarten-routine-vaccination-rates-continue-to-decline/
- PAHO, *PAHO calls for regional action as Americas lose measles elimination status* (November 2025)
